# Лабораторна робота № 7
## Логістична регресія. «Врятувати принцесу»


**Мета:** виконати бінарну класифікацію для датасету *Cure the Princess*, побудувати логістичну регресію, підібрати її параметри та порівняти з SVM і деревом рішень.

**Датасет:** [завантажити файл](https://drive.google.com/file/d/1ctefpJ5A0ZiwewxqLaC3RBeydLKLTgvH/view?usp=drive_link).

**Завдання:** допишіть код у клітинках із позначкою `TODO`, запустіть усі потрібні клітинки, після кожного етапу коротко прокоментуйте результати. Готові числові результати у шаблоні навмисно не наведено.

**Виконав:** Харченко Артьом, група ФІТ 3-10

https://www.kaggle.com/datasets/unmoved/cure-the-princess

### 1. Імпорт бібліотек

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score

### 2. Завантаження даних

Використайте **один** із двох способів. Якщо обираєте завантаження з комп'ютера, виконайте перший блок. Якщо KaggleHub — другий. Файл має назву `data.csv`.

In [ ]:
# Варіант 1. Завантаження файлу в Google Colab
# from google.colab import files
# files.upload()
# df = pd.read_csv('data.csv')
# df.head()

In [ ]:
# Варіант 2. Завантаження датасету через KaggleHub
import kagglehub
import os

path = kagglehub.dataset_download('unmoved/cure-the-princess')
df = pd.read_csv(os.path.join(path, 'data.csv'))
df.head()

### Опис даних

Цільова змінна **`Cured`**: `1` — принцесу вилікувано; `0` — не вилікувано. Інші стовпці — характеристики магічних інгредієнтів.

1. **Phoenix Feather** (Перо фенікса) — перо міфічної птиці фенікса, який згорає, а потім відроджується з власного попелу. Це символізує відродження, вічне життя та силу.

2. **Unicorn Horn** (Ріг єдинорога) — ріг міфічної істоти єдинорога, який часто асоціюється з чистотою та цілющою силою. У багатьох легендах вважалося, що ріг єдинорога має магічні властивості, зокрема здатність лікувати отруєння.

3. **Dragon's Blood** (Кров дракона) — кров дракона, міфічної істоти, часто вважається дуже потужним магічним зіллям або елементом. Це може символізувати силу, енергію чи небезпеку.

4. **Mermaid Tears**(Сльози русалки) — сльози міфічної істоти — русалки. У різних культурах їх вважають рідкісним і дуже цінним елементом, що має магічні властивості, часто пов'язані з любов'ю або трагічними подіями.

5. **Fairy Dust** (Пилок феї) — магічний пилок, що належить феям у народних казках. Зазвичай він використовується для чарівних перетворень, польотів або інших магічних дій.

6. **Goblin Toes** (Палець гобліна) — палець гобліна, вигаданої істоти з міфів, що зазвичай асоціюється з хитрістю і злісними чарами. Цей елемент може мати магічну силу або використовуватися в темних чарах.

7. **Witch's Brew** (Зілля відьми) — зілля, яке варить відьма, часто містить магічні або отруйні інгредієнти. Це часто асоціюється з темною магією або чарами.

8. **Griffin Claw** (Лапа грифона) — лапа грифона, міфічної істоти, що є поєднанням орла і лева. Грифон символізує силу та відвагу, а його лапа може вважатися потужним магічним предметом.

9. **Troll Hair** (Волосся троля) — волосся троля, ще однієї міфічної істоти, часто описуваної як агресивна та неприємна істота. Волосся троля може бути використане в магічних або темних ритуалах.

10. **Kraken Ink** (Чорнила кракена) — чорнила, що отримуються від кракена, гігантської морської істоти. У фентезі вони можуть використовуватись для темних магічних ритуалів або для створення важливих символів.

11. **Minotaur Horn** (Ріг минотавра) — ріг минотавра, міфічного чудовиська з головою бика і тілом людини. Ріг минотавра може символізувати силу і агресію, і в магічному контексті бути потужним інгредієнтом.

12. **Basilisk Scale** (Луска базиліска) — луска базиліска, міфічної істоти, яка може вбивати своїм поглядом. Луска базиліска у фентезі часто використовується як компонент у потужних зіллях або зброї.

13. **Chimera Fang** (Зуб хімери) — зуб хімери, міфічної істоти, яка поєднує різні тварини, такі як лев, коза і змія. Зуб хімери часто використовується в магічних зіллях або як символ сили та багатофункціональності.

14. **Cured** (Вилікувана) — цільова змінна: 1 — принцесу вилікувано, 0 — не вилікувано.

### 3. Первинний огляд датасету

Дослідіть структуру та розміри даних, пропущені значення, дублікати й описову статистику. Дайте короткий висновок про якість даних.

In [ ]:
display(df.head())
df.info()
print("Розмір:", df.shape)

In [ ]:
print("Пропусків:", df.isnull().sum().sum())
print("Дублікатів:", df.duplicated().sum())

In [ ]:
df.describe()

**Висновок за первинним оглядом:**
У датасеті 2338 записів і 14 стовпців: 13 ознак (інгредієнти, все числа) і цільова змінна Cured. Пропусків і дублікатів немає, тобто дані чисті. Значення всіх ознак від 1 до 40-45, але в різних ознак різні середні (наприклад, у Troll Hair близько 17, а в Unicorn Horn близько 11), тому стандартизація буде потрібна.

### 4. Поділ даних на навчальну й тестову вибірки

Відокремте `Cured` від ознак. Поділіть дані у співвідношенні **80% / 20%** із `random_state=42`.

**Важливо:** дослідження кореляцій і викидів, а також обчислення параметрів стандартизації виконуйте **тільки на навчальній вибірці**.

In [ ]:
X = df.drop(columns="Cured")
y = df["Cured"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train:", X_train.shape, "| Test:", X_test.shape)

### 5. Баланс класів на навчальній вибірці

In [ ]:
y_train.value_counts()

In [ ]:
sns.countplot(x=y_train)
plt.title("Класи в навчальній вибірці")
plt.show()

**Висновок:**
Класи майже порівну: у train 946 записів класу 0 і 924 класу 1. Тобто дисбалансу немає, і Accuracy можна використовувати як метрику. Якби одного класу було набагато більше, модель могла б просто передбачати його і мати високу точність, нічого не навчившись.

### 6. Дослідження викидів на навчальних даних

Побудуйте діаграми розмаху для ознак `X_train`. За правилом **1,5 IQR** знайдіть кількість потенційних викидів у кожній ознаці.

На основному етапі **не видаляйте викиди автоматично**. За бажанням пізніше проведіть окремий експеримент із видаленням тільки з навчальної вибірки.

In [ ]:
X_train.boxplot(figsize=(13, 5), rot=90)
plt.title("Boxplot ознак (train)")
plt.show()

In [ ]:
q1 = X_train.quantile(0.25)
q3 = X_train.quantile(0.75)
iqr = q3 - q1

outliers = ((X_train < q1 - 1.5 * iqr) | (X_train > q3 + 1.5 * iqr)).sum()
outliers.sort_values(ascending=False)

**Висновок про викиди:**
Викидів небагато. Найбільше у Witch's Brew (33), далі Unicorn Horn (20), Minotaur Horn (18) і Chimera Fang (13), в Mermaid Tears і Troll Hair їх немає зовсім. Це всього кілька відсотків від 1870 рядків train.

Без перевірки видаляти їх не варто: значення виглядають як нормальні, просто великі, це може бути справжня особливість даних, а не помилка. До того ж даних небагато, і видалення рядків зменшить навчальну вибірку.

### 7. Кореляційний аналіз на навчальній вибірці

1. Побудуйте кореляційну матрицю ознак **за модулем** (`.abs()`).
2. Відобразіть її у вигляді `heatmap` із **замаскованим верхнім трикутником** (`np.triu`).
3. Окремо знайдіть кореляцію ознак із цільовою змінною `Cured` **тільки для навчальної вибірки**.

In [ ]:
mtx = X_train.corr().abs()
mask = np.triu(np.ones_like(mtx, dtype=bool))

plt.figure(figsize=(11, 9))
sns.heatmap(mtx, mask=mask, annot=True, fmt=".2f", cmap="Reds", vmin=0, vmax=1)
plt.title("Кореляція ознак за модулем (train)")
plt.show()

In [ ]:
train_full = pd.concat([X_train, y_train], axis=1)
train_full.corr()["Cured"].drop("Cured").sort_values(ascending=False)

**Висновок про кореляції:**
Між ознаками кореляції невеликі, найбільша у пари Mermaid Tears і Kraken Ink (0.49), далі Mermaid Tears і Witch's Brew (0.34), Fairy Dust і Griffin Claw (0.34), Phoenix Feather і Troll Hair (0.33). Сильно корельованих ознак (більше 0.8) немає, тому нічого видаляти не треба.

З Cured найсильніше пов'язані Troll Hair (0.49) і Witch's Brew (0.25), у решти ознак кореляція майже нульова (менше 0.06). Але це тільки лінійні зв'язки, пізніше побачимо, що нелінійні моделі знаходять у даних більше.

### 8. Стандартизація ознак

Скористайтеся `StandardScaler`. Для `X_train` — `fit_transform()`, для `X_test` — тільки `transform()`.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### 9. Логістична регресія

Навчіть `LogisticRegression` на стандартизованих навчальних даних. Оцініть модель на тестовій вибірці: **Accuracy**, `classification_report`.

In [ ]:
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print(classification_report(y_test, y_pred))

### 10. Матриця помилок і приклади прогнозів

Побудуйте `confusion_matrix(y_test, y_pred)` та **теплову карту** (`sns.heatmap` із `annot=True`, `fmt='d'`). Порівняйте справжні й передбачені класи для **10 випадкових** тестових спостережень.

In [ ]:
conf = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(conf, annot=True, fmt="d", cmap="Blues")
plt.title("Матриця помилок, логістична регресія")
plt.xlabel("Прогноз")
plt.ylabel("Факт")
plt.show()

In [ ]:
compare = pd.DataFrame({"Справжній клас": y_test.values, "Передбачений клас": y_pred}, index=y_test.index)
compare.sample(10, random_state=42)

**Висновок щодо якості логістичної регресії:**
Accuracy 0.893, тобто модель правильно вгадала 89% з 468 тестових записів. Помилок небагато і майже порівну: 24 здорових записи передбачено як вилікувані і 26 вилікуваних як невилікувані (з матриці помилок 207 + 211 вгадано правильно). Precision і recall для обох класів близько 0.89-0.90.

Тут класи збалансовані, тому Accuracy підходить, але однієї метрики все одно мало: Accuracy не показує, які саме помилки робить модель, для цього потрібні матриця помилок, precision і recall.

### 11. Підбір гіперпараметрів

Задайте набори параметрів для `LogisticRegression`. Проведіть підбір через **GridSearchCV** та **RandomizedSearchCV** із `cv=3`, `scoring='accuracy'`. Порівняйте найкращі параметри та отриману якість.

*Підказка:* дослідіть параметри `C`, `solver`, `max_iter`. Підбір виконуйте на **навчальній** вибірці.

In [ ]:
param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],
    "solver": ["lbfgs", "liblinear"],
    "max_iter": [100, 200, 500],
}

param_dist = {
    "C": [0.001, 0.01, 0.1, 1, 10, 100, 1000],
    "solver": ["lbfgs", "liblinear", "newton-cg"],
    "max_iter": [100, 200, 500, 1000],
}

In [ ]:
grid = GridSearchCV(LogisticRegression(), param_grid, cv=3, scoring="accuracy")
grid.fit(X_train_scaled, y_train)

print("GridSearchCV, найкращі параметри:", grid.best_params_)
print("Accuracy на кросвалідації:", round(grid.best_score_, 4))

In [ ]:
random_search = RandomizedSearchCV(LogisticRegression(), param_dist, n_iter=15, cv=3,
                                   scoring="accuracy", random_state=42)
random_search.fit(X_train_scaled, y_train)

print("RandomizedSearchCV, найкращі параметри:", random_search.best_params_)
print("Accuracy на кросвалідації:", round(random_search.best_score_, 4))

In [ ]:
best_model = grid.best_estimator_
y_pred_best = best_model.predict(X_test_scaled)

print("Accuracy на тесті:", round(accuracy_score(y_test, y_pred_best), 4))
print(classification_report(y_test, y_pred_best))

**Висновок про підбір параметрів:**
GridSearchCV вибрав C = 10, solver = lbfgs, max_iter = 100. RandomizedSearchCV вибрав C = 1000, solver = liblinear, max_iter = 100, тобто параметри різні. Але Accuracy на кросвалідації однакова, 0.867, а на тесті з найкращою моделлю Accuracy 0.893, як і в базової.

Тобто підбір параметрів якість не покращив. Логістична регресія проста, і її можливості обмежує сама модель (вона лінійна), а не параметри.

### 12. Порівняння трьох моделей

Побудуйте також **метод опорних векторів (`SVC`)** і **дерево рішень (`DecisionTreeClassifier`)**. Порівняйте ці дві моделі з логістичною регресією за **Accuracy** та **F1-score** на однаковій тестовій вибірці. Оберіть найкращу модель і поясніть вибір.

In [ ]:
svc = SVC()
tree = DecisionTreeClassifier(random_state=42)

models = {"Logistic Regression": model, "SVC": svc, "Decision Tree": tree}

rows = []
for name, m in models.items():
    m.fit(X_train_scaled, y_train)
    pred = m.predict(X_test_scaled)
    rows.append({"Модель": name, "Accuracy": accuracy_score(y_test, pred), "F1-score": f1_score(y_test, pred)})

results = pd.DataFrame(rows).sort_values("F1-score", ascending=False).reset_index(drop=True)
display(results.round(4))
print("Найкраща модель:", results.loc[0, "Модель"])

### 13. Висновки до лабораторної роботи

1. Дані чисті: 2338 записів, 13 числових ознак, пропусків і дублікатів немає, класи майже порівну (946 і 924 у train).
2. У train викидів небагато (найбільше у Witch's Brew, 33), їх не видаляли. Сильно корельованих ознак немає, найбільша кореляція між ознаками 0.49. З Cured найсильніше пов'язані Troll Hair (0.49) і Witch's Brew (0.25).
3. Базова логістична регресія на тесті дала Accuracy 0.893 і F1 близько 0.89.
4. Підбір гіперпараметрів (GridSearchCV і RandomizedSearchCV) метрики не покращив: Accuracy на тесті залишилась 0.893.
5. З трьох моделей найкращою виявився SVC: Accuracy 0.959 і F1 0.960. Логістична регресія дала 0.893 / 0.894, дерево рішень найгірше, 0.865 / 0.866 (воно ще й повністю підлаштувалось під train, на ньому точність 1.0, тобто перенавчилось). SVC краща, бо розділяє класи нелінійно (за замовчуванням ядро rbf), а лінійні кореляції в даних слабкі, тому логістична регресія видає менше.

**Загальний висновок:** для цього датасету найкраще брати SVC, а логістична регресія дає хороший, але гірший результат.

---
**Перед здачею:** перевірте, що всі потрібні клітинки виконуються послідовно, а результати й висновки присутні у файлі `.ipynb`.